# Άσκηση 1: Εξερεύνηση και Έλεγχος Ποιότητας Δεδομένων — Superstore Dataset
## Το Σενάριο (Business Problem)

Είστε αναλυτής δεδομένων σε μια αμερικανική εταιρεία λιανικής πώλησης με τρεις κατηγορίες προϊόντων: **Furniture**, **Office Supplies** και **Technology**. Διαθέτετε ιστορικά δεδομένα πωλήσεων και καλείστε να απαντήσετε:

> *"Τα δεδομένα μας είναι αξιόπιστα; Ποιες πωλήσεις ή κατηγορίες δημιουργούν ζημία;"*

**Dataset:** [Sample Superstore — Kaggle (vivek468)](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final)  
**Αρχείο:** `SampleSuperstore.csv` — 9.994 παραγγελίες, 13 στήλες

Σύμφωνα με το μοντέλο **Herbert Simon**, βρισκόμαστε στη φάση:
- **Intelligence:** Φόρτωση, κατανόηση δομής, εντοπισμός προβλημάτων ποιότητας δεδομένων
- **Design:** Ποιες διορθώσεις χρειάζονται πριν οποιαδήποτε ανάλυση;

## Στήλες του Dataset

| Στήλη | Τύπος | Περιγραφή |
|---|---|---|
| `Ship Mode` | Κατηγορικό | Τρόπος αποστολής παραγγελίας |
| `Segment` | Κατηγορικό | Τμήμα πελατών (Consumer, Corporate, Home Office) |
| `Country` | Κατηγορικό | Χώρα (πάντα United States) |
| `City` | Κατηγορικό | Πόλη παράδοσης |
| `State` | Κατηγορικό | Πολιτεία παράδοσης |
| `Postal Code` | Αριθμητικό* | Ταχυδρομικός κωδικός — *προσοχή: τεχνικά κατηγορικό* |
| `Region` | Κατηγορικό | Γεωγραφική περιοχή (East, West, Central, South) |
| `Category` | Κατηγορικό | Κατηγορία προϊόντος (Furniture, Office Supplies, Technology) |
| `Sub-Category` | Κατηγορικό | Υποκατηγορία προϊόντος (17 τιμές) |
| `Sales` | Αριθμητικό | Αξία πώλησης (USD) |
| `Quantity` | Ακέραιος | Ποσότητα τεμαχίων |
| `Discount` | Αριθμητικό | Ποσοστό έκπτωσης (0 = 0%, 0.8 = 80%) |
| `Profit` | Αριθμητικό | Κέρδος/ζημία (USD) — **μπορεί να είναι αρνητικό!** |

> **Σημείωση:** Το dataset **δεν έχει** `Order ID` ή `Customer ID` στήλες — δεν μπορούμε να ομαδοποιήσουμε ανά παραγγελία απευθείας.

---
## Βήμα 1: Εγκατάσταση & Εισαγωγή Βιβλιοθηκών

In [ ]:
pip install pandas matplotlib seaborn numpy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ρυθμίσεις εμφάνισης
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid', palette='muted')

print("Βιβλιοθήκες φορτώθηκαν επιτυχώς!")

---
## Βήμα 2: Φόρτωση Δεδομένων

In [ ]:
df = pd.read_csv('SampleSuperstore.csv')

print(f"✔ Dataset φορτώθηκε: {df.shape[0]:,} γραμμές × {df.shape[1]} στήλες")
print(f"Στήλες: {df.columns.tolist()}")

---
## Βήμα 3: Πρώτη Ματιά στα Δεδομένα

### 3α. Πρώτες εγγραφές

In [ ]:
df.head()

### 3β. Τύποι δεδομένων και μη-κενές τιμές

In [ ]:
df.info()

### 3γ. Βασικά στατιστικά αριθμητικών στηλών

In [ ]:
df.describe()

---
## Βήμα 4: Έλεγχος Ποιότητας Δεδομένων (Data Quality Check)

Ο έλεγχος ποιότητας εξετάζει **4 βασικά προβλήματα**:

| # | Πρόβλημα | Τι ψάχνουμε |
|---|---|---|
| 1 | **Ελλείπουσες τιμές** | `NaN` — κενά κελιά |
| 2 | **Διπλότυπες εγγραφές** | Ολόιδιες γραμμές που μετρήθηκαν δύο φορές |
| 3 | **Λανθασμένοι τύποι δεδομένων** | Π.χ. αριθμός που πρέπει να είναι κείμενο |
| 4 | **Ακραίες/άκυρες τιμές** | Αρνητικές πωλήσεις, εκπτώσεις > 100%, κ.λπ. |

### 4α. Ελλείπουσες Τιμές (Missing Values)

In [ ]:
# Μετρήστε NaN ανά στήλη και υπολογίστε ποσοστό
missing = df.isnull().sum()
missing_pct = missing / len(df) * 100

missing_df = pd.DataFrame({
    'Κενές τιμές': missing,
    'Ποσοστό (%)': missing_pct
}).sort_values('Κενές τιμές', ascending=False)

print(f"Σύνολο στηλών: {len(df.columns)}")
print(f"Στήλες με NaN: {(missing > 0).sum()}")
print()
missing_df

### 4β. Διπλότυπες Εγγραφές (Duplicates)

Μια **διπλότυπη εγγραφή** είναι γραμμή που εμφανίζεται δύο ή περισσότερες φορές στο dataset. Μπορεί να δημιουργηθεί από:
- Σφάλμα κατά την εισαγωγή δεδομένων (double-entry)
- Διπλή εξαγωγή από σύστημα
- Συνένωση αρχείων χωρίς επαλήθευση

In [ ]:
n_duplicates = df.duplicated().sum()
print(f"Διπλότυπες εγγραφές: {n_duplicates} ({n_duplicates/len(df)*100:.2f}%)")
print()

# Εμφάνιση των διπλότυπων
if n_duplicates > 0:
    dupes = df[df.duplicated(keep=False)].sort_values(df.columns.tolist())
    print(f"Παράδειγμα διπλότυπων (πρώτα {min(6, len(dupes))} από {len(dupes)} γραμμές):")
    dupes.head(6)

In [ ]:
# Αφαίρεση διπλότυπων και επαλήθευση
df_clean = df.drop_duplicates()
print(f"Πριν:  {len(df):,} γραμμές")
print(f"Μετά:  {len(df_clean):,} γραμμές")
print(f"Αφαιρέθηκαν: {len(df) - len(df_clean)} γραμμές")

### 4γ. Λανθασμένοι Τύποι Δεδομένων

Το `.info()` έδειξε ότι η στήλη `Postal Code` έχει τύπο `int64`.  
Ένας ταχυδρομικός κωδικός **δεν είναι αριθμός** — δεν έχει νόημα να υπολογίσουμε τον μέσο ταχυδρομικό κωδικό! Πρέπει να είναι `string` (κατηγορικό).

Επίσης, οι στήλες `Ship Mode`, `Segment`, `Region`, `Category`, `Sub-Category` είναι κατάλληλες να μετατραπούν σε `category` dtype για εξοικονόμηση μνήμης.

In [ ]:
# Διόρθωση τύπων δεδομένων
df_clean = df_clean.copy()

# Postal Code: int → string (zero-padded στα 5 ψηφία, π.χ. 01040)
df_clean['Postal Code'] = df_clean['Postal Code'].astype(str).str.zfill(5)

# Κατηγορικές στήλες → category (εξοικονόμηση μνήμης)
cat_cols = ['Ship Mode', 'Segment', 'Country', 'Region', 'Category', 'Sub-Category']
for col in cat_cols:
    df_clean[col] = df_clean[col].astype('category')

print("Τύποι δεδομένων μετά τη διόρθωση:")
print(df_clean.dtypes)
print()

# Σύγκριση μνήμης πριν/μετά
mem_before = df.memory_usage(deep=True).sum() / 1024
mem_after  = df_clean.memory_usage(deep=True).sum() / 1024
print(f"Μνήμη πριν:  {mem_before:.1f} KB")
print(f"Μνήμη μετά:  {mem_after:.1f} KB  (−{mem_before-mem_after:.1f} KB)")

### 4δ. Έλεγχος Ακραίων/Άκυρων Τιμών (Outliers & Invalid Values)

Εξετάζουμε τρεις αριθμητικές στήλες για λογική εγκυρότητα:

| Στήλη | Αναμενόμενο εύρος | Τι ελέγχουμε |
|---|---|---|
| `Sales` | > 0 | Αρνητικές ή μηδενικές πωλήσεις |
| `Discount` | 0.0 – 1.0 | Εκτός εύρους τιμές |
| `Profit` | Οποιαδήποτε | Αρνητικά κέρδη (ζημία) — πιθανό επιχειρηματικό πρόβλημα |

In [ ]:
# --- Έλεγχος Sales ---
invalid_sales = df_clean[df_clean['Sales'] <= 0]
print(f"Εγγραφές με Sales ≤ 0:  {len(invalid_sales)}")

# --- Έλεγχος Discount ---
invalid_disc = df_clean[(df_clean['Discount'] < 0) | (df_clean['Discount'] > 1)]
print(f"Εγγραφές με Discount εκτός [0,1]:  {len(invalid_disc)}")
print(f"Εύρος Discount: {df_clean['Discount'].min():.2f} – {df_clean['Discount'].max():.2f}")
print(f"Μοναδικές τιμές Discount: {sorted(df_clean['Discount'].unique())}")

# --- Έλεγχος Profit (αρνητικά) ---
negative_profit = df_clean[df_clean['Profit'] < 0]
print(f"\nΕγγραφές με αρνητικό Profit (ζημία): {len(negative_profit)} "
      f"({len(negative_profit)/len(df_clean)*100:.1f}%)")
print(f"Συνολική ζημία: ${negative_profit['Profit'].sum():,.2f}")
print(f"Μέγιστη ζημία σε μία εγγραφή: ${negative_profit['Profit'].min():,.2f}")

#### Αρνητικό Profit ανά Κατηγορία

Τα αρνητικά κέρδη δεν είναι απαραίτητα "λάθος δεδομένο" — μπορεί να αντικατοπτρίζουν υψηλές εκπτώσεις. Ας δούμε ποια κατηγορία έχει τις περισσότερες ζημιογόνες πωλήσεις.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Ανάλυση Ζημιογόνων Πωλήσεων (Profit < 0)', fontsize=13, fontweight='bold')

# --- Αριστερά: αριθμός ζημιογόνων γραμμών ανά Category ---
loss_by_cat = negative_profit.groupby('Category', observed=True).size()
profit_by_cat = df_clean[df_clean['Profit'] >= 0].groupby('Category', observed=True).size()

x = np.arange(len(loss_by_cat))
w = 0.38
axes[0].bar(x - w/2, profit_by_cat.values, w, label='Κερδοφόρες', color='steelblue')
axes[0].bar(x + w/2, loss_by_cat.values,   w, label='Ζημιογόνες', color='tomato')
axes[0].set_xticks(x)
axes[0].set_xticklabels(loss_by_cat.index)
axes[0].set_ylabel('Αριθμός εγγραφών')
axes[0].set_title('Κερδοφόρες vs Ζημιογόνες ανά Κατηγορία')
axes[0].legend()

# --- Δεξιά: συνολικό profit ανά Category ---
total_profit = df_clean.groupby('Category', observed=True)['Profit'].sum()
colors = ['tomato' if v < 0 else 'steelblue' for v in total_profit.values]
axes[1].bar(total_profit.index, total_profit.values, color=colors)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_ylabel('Συνολικό Profit ($)')
axes[1].set_title('Συνολικό Κέρδος ανά Κατηγορία')
for i, (cat, val) in enumerate(total_profit.items()):
    axes[1].text(i, val + (500 if val >= 0 else -3000), f'${val:,.0f}',
                 ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nΣυνολικό Profit ανά Category:")
print(total_profit.to_string())

#### Σχέση Discount — Profit

Μια κοινή αιτία ζημίας είναι η **υπερβολική έκπτωση**. Ας εξετάσουμε αν υπάρχει συστηματική σχέση.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Σχέση Discount — Profit', fontsize=13, fontweight='bold')

# --- Αριστερά: Scatter plot Discount vs Profit ---
colors_scatter = ['tomato' if p < 0 else 'steelblue' for p in df_clean['Profit']]
axes[0].scatter(df_clean['Discount'], df_clean['Profit'],
                c=colors_scatter, alpha=0.3, s=8)
axes[0].axhline(0, color='black', linewidth=1, linestyle='--')
axes[0].set_xlabel('Discount')
axes[0].set_ylabel('Profit ($)')
axes[0].set_title('Scatter: Discount vs Profit')

# --- Δεξιά: Μέσο Profit ανά επίπεδο Discount ---
avg_profit_by_disc = df_clean.groupby('Discount')['Profit'].mean().reset_index()
bar_colors = ['tomato' if v < 0 else 'steelblue' for v in avg_profit_by_disc['Profit']]
axes[1].bar([str(d) for d in avg_profit_by_disc['Discount']],
             avg_profit_by_disc['Profit'], color=bar_colors)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xlabel('Discount')
axes[1].set_ylabel('Μέσο Profit ($)')
axes[1].set_title('Μέσο Profit ανά επίπεδο Discount')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

r = df_clean['Discount'].corr(df_clean['Profit'])
print(f"Pearson r (Discount, Profit) = {r:.3f}")

---
## Βήμα 5: Κατανομές Αριθμητικών Μεταβλητών

Εξετάζουμε την κατανομή των τεσσάρων αριθμητικών μεταβλητών: `Sales`, `Quantity`, `Discount`, `Profit`.

In [ ]:
num_cols = ['Sales', 'Quantity', 'Discount', 'Profit']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Κατανομές Αριθμητικών Μεταβλητών', fontsize=13, fontweight='bold')

for i, col in enumerate(num_cols):
    data = df_clean[col]
    μ, σ = data.mean(), data.std()
    skew = data.skew()

    # Ιστόγραμμα (πάνω γραμμή)
    ax_hist = axes[0, i]
    ax_hist.hist(data, bins=40, color='steelblue', alpha=0.75, edgecolor='white')
    ax_hist.axvline(μ, color='red',    linestyle='-',  linewidth=1.5, label=f'Μέσος={μ:.1f}')
    ax_hist.axvline(data.median(), color='orange', linestyle='--', linewidth=1.5,
                    label=f'Διάμεσος={data.median():.1f}')
    ax_hist.set_title(f'{col}\nskewness={skew:.2f}', fontsize=10)
    ax_hist.set_xlabel(col, fontsize=8)
    ax_hist.legend(fontsize=7)

    # Boxplot (κάτω γραμμή)
    ax_box = axes[1, i]
    ax_box.boxplot(data, vert=True, patch_artist=True,
                   boxprops=dict(facecolor='lightsteelblue'),
                   medianprops=dict(color='red', linewidth=2))
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr = q3 - q1
    n_outliers = ((data < q1 - 1.5*iqr) | (data > q3 + 1.5*iqr)).sum()
    ax_box.set_title(f'IQR={iqr:.1f}\nOutliers: {n_outliers}', fontsize=9)
    ax_box.set_xticks([])

plt.tight_layout()
plt.show()

---
## Βήμα 6: Εξερεύνηση Κατηγορικών Μεταβλητών

### 6α. Πληθυσμός κατηγοριών (cardinality)

In [ ]:
# Μοναδικές τιμές ανά κατηγορική στήλη
all_cat = ['Ship Mode', 'Segment', 'Country', 'City', 'State', 'Region', 'Category', 'Sub-Category']

cardinality = pd.DataFrame({
    'Μοναδικές τιμές': [df_clean[c].nunique() for c in all_cat],
    'Παραδείγματα': [', '.join(df_clean[c].unique()[:4].astype(str)) + ('...' if df_clean[c].nunique() > 4 else '')
                     for c in all_cat]
}, index=all_cat)
cardinality

### 6β. Κατανομή πωλήσεων ανά βασικές κατηγορίες

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Κατανομή Εγγραφών ανά Κατηγορική Μεταβλητή', fontsize=13, fontweight='bold')

for ax, col in zip(axes, ['Category', 'Segment', 'Region']):
    counts = df_clean[col].value_counts()
    bars = ax.bar(counts.index, counts.values, color='steelblue', edgecolor='white')
    ax.set_title(col, fontweight='bold')
    ax.set_ylabel('Αριθμός εγγραφών')
    ax.tick_params(axis='x', rotation=15)
    # Αριθμός και ποσοστό πάνω από κάθε μπάρα
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f'{val:,}\n({val/len(df_clean)*100:.1f}%)',
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

### 6γ. Sub-Category: πωλήσεις και κερδοφορία

Οι 17 υποκατηγορίες έχουν πολύ διαφορετική συμπεριφορά — ποιες είναι κερδοφόρες και ποιες ζημιογόνες;

In [ ]:
subcat_summary = df_clean.groupby('Sub-Category', observed=True).agg(
    Πωλήσεις=('Sales', 'sum'),
    Κέρδος=('Profit', 'sum'),
    Εγγραφές=('Sales', 'count')
).sort_values('Κέρδος', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Sub-Category: Συνολικές Πωλήσεις & Κέρδος', fontsize=13, fontweight='bold')

# Πωλήσεις
axes[0].barh(subcat_summary.index, subcat_summary['Πωλήσεις'], color='steelblue')
axes[0].set_xlabel('Συνολικές Πωλήσεις ($)')
axes[0].set_title('Πωλήσεις ανά Sub-Category')

# Κέρδος
colors = ['tomato' if v < 0 else 'seagreen' for v in subcat_summary['Κέρδος']]
axes[1].barh(subcat_summary.index, subcat_summary['Κέρδος'], color=colors)
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_xlabel('Συνολικό Κέρδος ($)')
axes[1].set_title('Κέρδος ανά Sub-Category\n(κόκκινο = ζημία)')

plt.tight_layout()
plt.show()

print("\nΖημιογόνες Sub-Categories:")
print(subcat_summary[subcat_summary['Κέρδος'] < 0][['Πωλήσεις', 'Κέρδος', 'Εγγραφές']].to_string())

---
## Σύνοψη: Αποτελέσματα Ελέγχου Ποιότητας

### Τι βρήκαμε:

| Έλεγχος | Αποτέλεσμα | Χρειάζεται Ενέργεια; |
|---|---|---|
| **Ελλείπουσες τιμές (NaN)** | 0 NaN σε όλες τις στήλες | ✅ Όχι |
| **Διπλότυπες εγγραφές** | 17 διπλότυπα (0.17%) | ✅ Αφαιρέθηκαν με `drop_duplicates()` |
| **Τύπος `Postal Code`** | `int64` αντί `str` | ✅ Διορθώθηκε σε `str.zfill(5)` |
| **Κατηγορικές στήλες** | `object` αντί `category` | ✅ Διορθώθηκαν (εξοικονόμηση μνήμης) |
| **Τιμές `Sales`** | Όλες > 0 | ✅ Εντάξει |
| **Τιμές `Discount`** | 0.0 – 0.8 (εντός [0,1]) | ✅ Εντάξει |
| **Αρνητικό `Profit`** | ~1.800 εγγραφές (~18%) | ⚠️ Επιχειρηματικό ζήτημα, όχι λάθος δεδομένου |

### Κύριο Εύρημα:
> Η κατηγορία **Furniture** (και ιδίως το Sub-Category **Tables**) παράγει **συστηματική ζημία** παρά τις υψηλές πωλήσεις. Η αιτία φαίνεται να είναι οι **υψηλές εκπτώσεις** (Discount ≥ 0.4), που οδηγούν σε αρνητικό Profit.

### Επόμενα Βήματα (Design Phase):
1. Αναλύστε βαθύτερα τη σχέση Discount–Profit ανά Sub-Category
2. Εξετάστε αν συγκεκριμένα `State` ή `City` συσσωρεύουν ζημίες
3. Χτίστε μοντέλο πρόβλεψης κερδοφορίας βάσει χαρακτηριστικών παραγγελίας